# Condition-classifier figures

Renders every figure for the WAT QPI condition classifier from the artifacts a
run leaves behind (`predictions.csv`, `metrics.json`, `attribution_stats.csv`,
`gradcam_maps.npz`, `ig_maps.npz`) - no model and no GPU needed, so re-running
the whole notebook takes seconds.

Everything you are likely to change - font, label names, class order, colours,
sizes - lives in the **Config** cell. Figures are written to `<run>/figures/`
as SVG + PNG. The SVG keeps text as text (`svg.fonttype = "none"`), so labels
stay editable in Illustrator or Inkscape.

To produce the inputs, run once from the repo root:

```
python -m phenotyping.supervised_classification.resnet_condition_classifier --mode finetune --target-size 224
python -m phenotyping.supervised_classification.attribution_maps --run-dir "<run folder>"
```

In [ ]:
# --- Config: everything you are likely to tweak -----------------------------
from pathlib import Path

RUN_DIR = Path(
    "C:/Users/anous/OneDrive - Johns Hopkins/2026_datanalysis/MISC_NONBC_NONFIBRO"
    "/WAT_AYAN/Total HWAT data/All_WAT_MIP/resnet_classification/resnet18_finetune"
)
FIGURE_DIR = RUN_DIR / "figures"

# Folder with the Roboto .ttf files (any folder of .ttf works).
FONT_DIR = Path("C:/Users/anous/Downloads/Roboto (1)")
FONT_NAME = "Roboto"
BASE_FONT_SIZE = 8

# Folder condition -> label printed on every figure.
DISPLAY_NAMES = {
    "D7_C1": "D7 C1", "D7_C3": "D7 C2", "D7_C4": "D7 C3", "D7_C5": "D7 C4",
    "D14_C1": "D14 C1", "D14_C3": "D14 C2", "D14_C4": "D14 C3", "D14_C5": "D14 C4",
}
# Row / column / panel order (chronological: day 7 before day 14).
CLASS_ORDER = ["D7_C1", "D7_C3", "D7_C4", "D7_C5",
               "D14_C1", "D14_C3", "D14_C4", "D14_C5"]

N_EXAMPLES = 3            # images per condition in the attribution panels
N_REPRESENTATIVE = 4      # images per condition in the overview figure
CAM_CMAP, CAM_ALPHA = "inferno", 0.55
IG_CMAP, IG_ALPHA = "magma", 0.85
CM_CMAP = "Blues"
DPI = 300

In [ ]:
# --- Setup: font, style, helpers --------------------------------------------
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from sklearn.metrics import confusion_matrix

# make the repo importable no matter where the notebook is launched from
REPO = Path.cwd()
while not (REPO / "phenotyping").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from phenotyping.supervised_classification.resnet_condition_classifier import load_image

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for ttf in sorted(FONT_DIR.glob("*.ttf")):
    font_manager.fontManager.addfont(str(ttf))
installed = {f.name for f in font_manager.fontManager.ttflist}
FAMILY = FONT_NAME if FONT_NAME in installed else "DejaVu Sans"
if FAMILY != FONT_NAME:
    print(f"{FONT_NAME} not found in {FONT_DIR} - falling back to {FAMILY}")

plt.rcParams.update({
    "font.family": FAMILY,
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": BASE_FONT_SIZE + 1,
    "axes.labelsize": BASE_FONT_SIZE,
    "xtick.labelsize": BASE_FONT_SIZE - 1,
    "ytick.labelsize": BASE_FONT_SIZE - 1,
    "legend.fontsize": BASE_FONT_SIZE - 1,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "figure.dpi": 110,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "svg.fonttype": "none",   # keep SVG text editable in Illustrator/Inkscape
})


def label(class_name):
    """Folder condition -> display label."""
    return DISPLAY_NAMES.get(class_name, class_name)


def save(fig, name):
    for extension in ("svg", "png"):
        fig.savefig(FIGURE_DIR / f"{name}.{extension}")
    print("saved", (FIGURE_DIR / f"{name}.svg").name)


def bare(ax, frame=True):
    """Image axes: no ticks, optional thin frame."""
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(frame)
        spine.set_linewidth(0.4)
        spine.set_edgecolor("0.6")
    return ax


print("font:", FAMILY)

In [ ]:
# --- Load the run -----------------------------------------------------------
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
splits = pd.read_csv(RUN_DIR / "image_splits.csv")
predictions = pd.read_csv(RUN_DIR / "predictions.csv").merge(
    splits[["filename", "path"]], on="filename", how="left"
)
prob_columns = [f"prob_{name}" for name in metrics["class_names"]]
predictions["confidence"] = predictions[prob_columns].max(axis=1)

test = predictions[predictions["split"] == "test"].copy()
SIZE = metrics["target_size"]
TICK_LABELS = [label(c) for c in CLASS_ORDER]

# images are re-read on demand and cached here so panels stay fast
_image_cache = {}


def image_for(row):
    if row.filename not in _image_cache:
        _image_cache[row.filename] = load_image(row.path, SIZE)
    return _image_cache[row.filename]


print(f"{len(test)} test images | "
      f"val {metrics['balanced_accuracy_val']:.3f} | "
      f"test {metrics['balanced_accuracy_test']:.3f} balanced accuracy")

## Figure 1 - representative images per condition

In [ ]:
fig, axs = plt.subplots(
    len(CLASS_ORDER), N_REPRESENTATIVE,
    figsize=(N_REPRESENTATIVE * 1.05, len(CLASS_ORDER) * 1.1),
)
for row, class_name in enumerate(CLASS_ORDER):
    subset = splits[(splits["class_name"] == class_name)
                    & (splits["split"] == "train")].head(N_REPRESENTATIVE)
    for col in range(N_REPRESENTATIVE):
        ax = bare(axs[row, col])
        if col < len(subset):
            ax.imshow(load_image(subset.iloc[col]["path"], SIZE),
                      cmap="gray", vmin=0, vmax=1)
        if col == 0:
            ax.set_ylabel(label(class_name), rotation=0, ha="right",
                          va="center", labelpad=5)
fig.tight_layout(h_pad=0.25, w_pad=0.15)
save(fig, "fig1_representative_images")

## Figure 2 - training and validation loss

In [ ]:
fig, ax = plt.subplots(figsize=(3.0, 2.2))
ax.plot(metrics["losses_train"], lw=1.0, color="0.25", label="train")
ax.plot(metrics["losses_val"], lw=1.0, color="C1", label="validation")
ax.axvline(metrics["best_epoch"], color="0.65", ls=":", lw=0.8)
ax.annotate(f"best epoch {metrics['best_epoch']}",
            xy=(metrics["best_epoch"], ax.get_ylim()[1]),
            xytext=(3, -2), textcoords="offset points",
            va="top", fontsize=BASE_FONT_SIZE - 1, color="0.4")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend(frameon=False)
save(fig, "fig2_training_loss")

## Figure 3 - confusion matrices

In [ ]:
def confusion_panel(frame, title, name):
    matrix = confusion_matrix(frame["true_class"], frame["predicted_class"],
                              labels=CLASS_ORDER)
    fraction = matrix / np.clip(matrix.sum(axis=1, keepdims=True), 1, None)

    fig, ax = plt.subplots(figsize=(3.4, 3.0))
    image = ax.imshow(fraction, cmap=CM_CMAP, vmin=0, vmax=1)
    for i in range(len(CLASS_ORDER)):
        for j in range(len(CLASS_ORDER)):
            ax.text(j, i, matrix[i, j], ha="center", va="center",
                    fontsize=BASE_FONT_SIZE - 1,
                    color="white" if fraction[i, j] > 0.55 else "0.2")
    ax.set_xticks(range(len(CLASS_ORDER)), TICK_LABELS, rotation=90)
    ax.set_yticks(range(len(CLASS_ORDER)), TICK_LABELS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for spine in ax.spines.values():
        spine.set_visible(False)
    bar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, ticks=[0, 0.5, 1])
    bar.set_label("fraction of true class")
    bar.outline.set_visible(False)
    save(fig, name)
    return matrix


confusion_panel(test, "Test", "fig3_confusion_matrix_test")
confusion_panel(predictions[predictions["split"] == "val"],
                "Validation", "fig3_confusion_matrix_val")

## Figure 4 - correct and misclassified test images

Top row: correct calls. Bottom row: errors, labelled `true -> predicted`.

In [ ]:
rng = np.random.default_rng(0)
correct_mask = test["correct"].astype(bool)
n_show = 6

fig, axs = plt.subplots(2, n_show, figsize=(n_show * 1.25, 3.0))
for row, (subset, row_title) in enumerate(
    [(test[correct_mask], "Correct"), (test[~correct_mask], "Misclassified")]
):
    subset = subset.sort_values("true_class", kind="stable")
    take = sorted(rng.choice(len(subset), size=min(n_show, len(subset)),
                             replace=False))
    for col in range(n_show):
        ax = bare(axs[row, col])
        if col >= len(take):
            ax.set_visible(False)
            continue
        item = subset.iloc[take[col]]
        ax.imshow(image_for(item), cmap="gray", vmin=0, vmax=1)
        title = (label(item.true_class) if row == 0
                 else f"{label(item.true_class)} $\\rightarrow$ {label(item.predicted_class)}")
        ax.set_title(title, fontsize=BASE_FONT_SIZE - 1,
                     color="0.2" if row == 0 else "C3", pad=2)
        if col == 0:
            ax.set_ylabel(row_title, rotation=90, va="center", labelpad=4)
fig.tight_layout(h_pad=0.6, w_pad=0.2)
save(fig, "fig4_example_predictions")

## Figures 5 and 6 - attribution panels

Grad-CAM (regional, `layer4`) and integrated gradients (pixel level), both read
from the caches written by `attribution_maps.py`. Each row shows the most
confident correct calls for that condition; if a condition has too few, the row
is topped up with its remaining images and those are flagged in red.

In [ ]:
gradcam_maps = np.load(RUN_DIR / "gradcam_maps.npz")
ig_maps = np.load(RUN_DIR / "ig_maps.npz")

picks = {}
for class_name in CLASS_ORDER:
    of_class = test[test["true_class"] == class_name]
    chosen = of_class[of_class["correct"]].nlargest(N_EXAMPLES, "confidence")
    if len(chosen) < N_EXAMPLES:
        chosen = pd.concat([chosen, of_class[~of_class["correct"]].nlargest(
            N_EXAMPLES - len(chosen), "confidence")])
    picks[class_name] = chosen


def attribution_panel(maps, cmap, alpha, title, name):
    fig, axs = plt.subplots(
        len(CLASS_ORDER), 2 * N_EXAMPLES,
        figsize=(2 * N_EXAMPLES * 1.05, len(CLASS_ORDER) * 1.15),
    )
    for row, class_name in enumerate(CLASS_ORDER):
        for col in range(2 * N_EXAMPLES):
            bare(axs[row, col]).set_visible(False)
        for i, item in enumerate(picks[class_name].itertuples()):
            ax_qpi, ax_map = axs[row, 2 * i], axs[row, 2 * i + 1]
            image = image_for(item)
            attribution = np.clip(maps[item.filename].astype(np.float32), 0, 1)
            for ax in (ax_qpi, ax_map):
                bare(ax).set_visible(True)
                ax.imshow(image, cmap="gray", vmin=0, vmax=1)
            # per-pixel alpha keeps weak attribution transparent
            ax_map.imshow(attribution, cmap=cmap, vmin=0, vmax=1,
                          alpha=attribution ** 1.5 * alpha)
            if not item.correct:
                ax_map.set_title("misclassified", fontsize=BASE_FONT_SIZE - 2,
                                 color="C3", pad=2)
            if row == 0:
                ax_qpi.set_title("QPI", fontsize=BASE_FONT_SIZE - 1, pad=2)
                if item.correct:
                    ax_map.set_title("attribution",
                                     fontsize=BASE_FONT_SIZE - 1, pad=2)
            if i == 0:
                ax_qpi.set_ylabel(label(class_name), rotation=0, ha="right",
                                  va="center", labelpad=5)
    fig.suptitle(title, fontsize=BASE_FONT_SIZE + 1)
    fig.tight_layout(h_pad=0.25, w_pad=0.15, rect=(0, 0, 0.92, 1))

    cax = fig.add_axes((0.935, 0.15, 0.012, 0.7))
    bar = fig.colorbar(
        plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap=cmap),
        cax=cax, ticks=[0, 1],
    )
    bar.set_label("attribution (scaled)", fontsize=BASE_FONT_SIZE - 1)
    bar.outline.set_visible(False)
    save(fig, name)


attribution_panel(gradcam_maps, CAM_CMAP, CAM_ALPHA,
                  "Grad-CAM", "fig5_gradcam_panel")
attribution_panel(ig_maps, IG_CMAP, IG_ALPHA,
                  "Integrated gradients", "fig6_integrated_gradients_panel")

## Figure 7 - attribution over lipid-dense pixels

Mean attribution over the top-5% refractive-index pixels divided by the mean
over the whole field. 1.0 (dashed) would mean the map is indifferent to
droplets.

In [ ]:
stats = pd.read_csv(RUN_DIR / "attribution_stats.csv")
column = "ig_lipid_enrichment"
values = [stats.loc[stats["true_class"] == c, column].dropna().to_numpy()
          for c in CLASS_ORDER]

fig, ax = plt.subplots(figsize=(4.0, 2.4))
boxes = ax.boxplot(values, widths=0.6, showfliers=False, patch_artist=True,
                   medianprops=dict(color="0.15", lw=1.0),
                   whiskerprops=dict(color="0.4", lw=0.6),
                   capprops=dict(color="0.4", lw=0.6))
for patch in boxes["boxes"]:
    patch.set(facecolor="#cfe2f3", edgecolor="0.4", lw=0.6)
jitter = np.random.default_rng(0)
for position, column_values in enumerate(values, start=1):
    ax.scatter(jitter.normal(position, 0.05, len(column_values)), column_values,
               s=3, color="0.25", alpha=0.7, zorder=3, linewidths=0)
ax.axhline(1.0, color="C3", ls="--", lw=0.8)
ax.set_xticks(range(1, len(CLASS_ORDER) + 1), TICK_LABELS, rotation=90)
ax.set_ylabel("IG in top 5% RI pixels\n(fold over field mean)")
ax.set_ylim(bottom=0.8)
save(fig, "fig7_lipid_enrichment")

print(stats.groupby("true_class")[column].median().round(2).to_string())